                 ┌─────────────────┐
                 │     Question    │
                 └────────┬────────┘
                          ↓
                 ┌─────────────────┐
                 │    Generate     │
                 └────────┬────────┘
                          ↓
                    Initial Answer
                          ↓
                 ┌─────────────────┐
                 │     Critic      │
                 └────────┬────────┘
                          ↓
                    Is pass=True?
                    /            \
                  YES             NO
                   ↓               ↓
              Return Answer     Revise
                                   ↓
                              New Answer
                                   ↓
                                Critic
                                   ↓
                                  ...

In [ ]:
from groq import Groq
import json
model = "openai/gpt-oss-20b"
api = "XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"
groq = Groq(api_key = api)

In [4]:
def call_llm(user_input , system_input):
    messages = [
        {
        'role':'system',
        'content':system_input
    },
    {'role':'user',
     'content':user_input}
    ]

    response = groq.chat.completions.create(
        model = model ,
        messages = messages,
        temperature=0.2


    )

    return response.choices[0].message.content

In [5]:
def generate(task):
    system_message =   """
You are an expert AI assistant.

Answer the user's question accurately,
clearly, and directly.

Do not invent facts.
If something is uncertain, say so.
"""
    return call_llm(user_input=task , system_input=system_message)

In [6]:
def critic(task , answer):
    system_prompt = """
You are a strict AI answer critic.

Your job is to evaluate an answer.

Check:

1. Correctness
2. Relevance
3. Completeness
4. Clarity
5. Hallucinations
6. Whether the answer actually answers the question

Return ONLY valid JSON:

{
    "pass": true or false,
    "score": number from 0 to 10,
    "issues": [
        "issue 1",
        "issue 2"
    ],
    "suggestions": [
        "suggestion 1",
        "suggestion 2"
    ]
}
"""
    prompet = f"Quesion:{task} , Answer{answer} "
    result = call_llm(user_input=prompet,system_input=system_prompt)
    
    try:
        return json.loads(result)

    except json.JSONDecodeError:

        # في حالة رجوع JSON داخل markdown
        result = result.replace(
            "```json",
            ""
        ).replace(
            "```",
            ""
        ).strip()

        try:
            return json.loads(result)

        except json.JSONDecodeError:

            return {
                "pass": False,
                "score": 0,
                "issues": [
                    "Critic returned invalid JSON"
                ],
                "suggestions": [
                    "Generate the critique again"
                ]
            }
    

In [7]:
def revise_answer(
    question: str,
    answer: str,
    critique: dict
) -> str:

    system_prompt = """
You are an expert answer reviser.

You receive:

- Original question
- Previous answer
- Critic feedback

Your job is to produce a better answer.

Fix every important issue mentioned by the critic.

Do not mention:
- the critic
- the previous answer
- the revision process

Return ONLY the final improved answer.
"""

    prompt = f"""
Original Question:

{question}


Previous Answer:

{answer}


Critic Feedback:

{json.dumps(critique, ensure_ascii=False, indent=2)}


Rewrite the answer and fix the problems.
"""
    return call_llm(user_input=prompt , system_input=system_prompt)

In [8]:
question = """
Explain why Transformer models use

Self-Attention instead of traditional RNNs.
"""

In [11]:
def self_critics_loop(question, max_iterations=3):

    print("=======================================")
    print("INITIAL ANSWER")
    print("=======================================")

    answer = generate(question)

    print(f"Question:\n{question}\n")
    print(f"Initial Answer:\n{answer}\n")

    for iteration in range(max_iterations):

        print("=======================================")
        print(f"ITERATION {iteration + 1}")
        print("=======================================")

        # 1. Critic
        critic_result = critic(
            task=question,
            answer=answer
        )

        print(f"PASS: {critic_result['pass']}")
        print(f"SCORE: {critic_result['score']}")
        print(f"ISSUES: {critic_result['issues']}")

        # 2. Stop if answer is good
        if critic_result["pass"]:
            print("\nAnswer passed the critic.")
            return answer

        # 3. Revise answer
        print("\nRevising answer...")

        answer = revise_answer(
            question=question,
            answer=answer,
            critique=critic_result
        )

    # Maximum iterations reached
    print("\nMaximum iterations reached.")

    return answer

In [12]:
loop = self_critics_loop(question=question)
print(loop)

INITIAL ANSWER
Question:

Explain why Transformer models use

Self-Attention instead of traditional RNNs.


Initial Answer:
**Why Transformers use Self‑Attention instead of RNNs**

| Feature | RNNs (LSTM/GRU) | Self‑Attention (Transformer) |
|---------|-----------------|------------------------------|
| **Processing order** | Sequential – one token at a time | Parallel – all tokens at once |
| **Context capture** | Limited to the hidden state that is propagated; long‑range dependencies are hard to learn (vanishing/exploding gradients) | Every token can directly attend to every other token, so the model can learn arbitrary‑length dependencies in a single layer |
| **Computational cost** | \(O(n \cdot d)\) per token, overall \(O(n^2 \cdot d)\) for a sequence of length \(n\) (because of the sequential loop) | \(O(n^2 \cdot d)\) for the whole sequence, but all operations are matrix‑multiplications that can be fully parallelized on GPUs/TPUs |
| **Parallelism** | No true parallelism – each 